# IMC 2025 — Adaptive Graph FAST submission

This is the **frozen fast pipeline** used after local validation. It deliberately does **not** run the expensive balanced rescue stage.

Attach these inputs before committing:
1. **Image Matching Challenge 2025 Ongoing** competition data.
2. Your private dataset created by `scripts/make_kaggle_bundle.py`.

Use a GPU and keep **Internet OFF** for the committed submission. The notebook writes `/kaggle/working/submission.csv`.


In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil, glob

print('Python:', sys.version)
try:
    import torch
    print('Torch:', torch.__version__)
    print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO CUDA')
except Exception as e:
    print(e)

INPUT = Path('/kaggle/input')
assert INPUT.exists(), 'This notebook is intended for Kaggle.'


In [ ]:
# Find the uploaded project bundle automatically.
project_candidates = []
for cfg in INPUT.rglob('configs/final.yaml'):
    project_candidates.append(cfg.parent.parent)
for cfg in INPUT.rglob('configs/fast.yaml'):
    project_candidates.append(cfg.parent.parent)
project_candidates = list(dict.fromkeys(project_candidates))
if not project_candidates:
    raise RuntimeError('Project bundle not found. Attach the dataset produced by scripts/make_kaggle_bundle.py')
PROJECT = project_candidates[0]
print('PROJECT =', PROJECT)

# Keep all pretrained weights offline.
os.environ['TORCH_HOME'] = str(PROJECT / 'models' / 'torch')
# The bundle stores the HuggingFace hub cache under models/hf (matches the dev flake).
# Fall back to models/huggingface only if a future bundle uses that name.
hf_home = PROJECT / 'models' / 'hf'
if not (hf_home / 'hub').exists() and (PROJECT / 'models' / 'huggingface' / 'hub').exists():
    hf_home = PROJECT / 'models' / 'huggingface'
os.environ['HF_HOME'] = str(hf_home)
os.environ['HF_HUB_CACHE'] = str(hf_home / 'hub')
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
print('HF cache =', os.environ['HF_HUB_CACHE'], '(exists:', (hf_home / 'hub').exists(), ')')

sys.path.insert(0, str(PROJECT / 'src'))
sys.path.insert(0, str(PROJECT / 'third_party' / 'LightGlue'))

In [ ]:
# Install the exact pycolmap API tested locally, from the uploaded offline wheel.
wheel_roots = [p for p in INPUT.rglob('wheels') if p.is_dir()]
wheel_roots += [PROJECT.parent / 'wheels']
wheel_roots = [p for p in wheel_roots if p.exists()]
wheelhouse = wheel_roots[0] if wheel_roots else None
print('wheelhouse =', wheelhouse)
if wheelhouse is None:
    raise RuntimeError('Offline wheelhouse not found in project bundle.')

def pip_install(pattern):
    xs = sorted(wheelhouse.glob(pattern))
    if not xs:
        raise RuntimeError(f'Missing wheel: {pattern}')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--force-reinstall', str(xs[-1])], check=True)

pip_install('pycolmap-3.13.0-*.whl')
# Install timm stack only if absent.
try:
    import timm
except Exception:
    pip_install('huggingface_hub-*.whl')
    pip_install('safetensors-*.whl')
    pip_install('timm-*.whl')

import pycolmap, timm, cv2, networkx, yaml
from lightglue import ALIKED, LightGlue
print('pycolmap:', pycolmap.__version__)
print('Offline imports OK')


In [ ]:
# Find the competition root. The hidden test folder is swapped in by Kaggle at scoring time.
roots = []
for sample in INPUT.rglob('sample_submission.csv'):
    parent = sample.parent
    if (parent / 'test').is_dir():
        roots.append(parent)
if not roots:
    raise RuntimeError('Competition input not found. Attach Image Matching Challenge 2025 Ongoing.')
# Prefer the ongoing competition path when present.
roots.sort(key=lambda p: ('ongoing' not in str(p).lower(), len(str(p))))
DATA = roots[0]
print('DATA =', DATA)
print('Visible test images:', len(list((DATA/'test').rglob('*.png'))))


In [ ]:
# Run the frozen, locally validated fast graph pipeline.
OUTROOT = Path('/kaggle/working/adaptive_graph_outputs')
if OUTROOT.exists():
    shutil.rmtree(OUTROOT)
config = PROJECT / 'configs' / ('final.yaml' if (PROJECT/'configs/final.yaml').exists() else 'fast.yaml')
cmd = [
    sys.executable, str(PROJECT/'scripts/run_pipeline.py'),
    '--mode', 'test',
    '--data-dir', str(DATA),
    '--config', str(config),
    '--run-name', 'final_test',
    '--output-root', str(OUTROOT),
]
print(' '.join(map(str, cmd)))
# run_pipeline.py runs as a subprocess and does NOT inherit the notebook's sys.path,
# so export the package + vendored LightGlue + offline weight caches via the environment.
env = os.environ.copy()
env['PYTHONPATH'] = os.pathsep.join([
    str(PROJECT / 'src'),
    str(PROJECT / 'third_party' / 'LightGlue'),
    env.get('PYTHONPATH', ''),
])
subprocess.run(cmd, cwd=PROJECT, check=True, env=env)

src = OUTROOT/'final_test'/'submission.csv'
dst = Path('/kaggle/working/submission.csv')
shutil.copy2(src, dst)
print('FINAL:', dst, 'bytes=', dst.stat().st_size)

In [ ]:
# Submission sanity checks.
import pandas as pd
sample = pd.read_csv(DATA/'sample_submission.csv')
sub = pd.read_csv('/kaggle/working/submission.csv')
print('sample rows:', len(sample), 'submission rows:', len(sub))
print(sub.head())
assert len(sub) == len(sample)
assert not sub[['dataset','scene','image','rotation_matrix','translation_vector']].isna().any().any()
print('SUBMISSION READY')
